# Discussion: re-subtyping autism mouse models through π

This is **not a Results section**. It is the worked application that appears in the Discussion, and it is
included as a notebook because it shows most clearly what a calibrated coupling is *for*.

**The setup.** Pagani et al. (2026) phenotype 20 autism mouse models by resting-state fMRI and split them
into two subtypes: **hyper-connected** (n = 9) and **hypo-connected** (n = 11). A clinician would ask
which human functional networks each subtype implicates. That is a mouse→human translation question, and
it is what π is for.

**What we do.** Route each subtype's mouse connectivity phenotype through π into human network space, and
compare against the observed human autism connectivity literature.

> ## ⚠️ Read the error history before trusting any number in here
>
> This analysis has been wrong twice, in two different ways, and both are instructive.
>
> **1. The subtype labels were inverted.** An earlier version had hyper- and hypo-connected the wrong way
> round. Every downstream conclusion was therefore backwards while looking perfectly coherent. A sign
> error does not announce itself. The labels are now verified directly from the data
> (`pagani_subtype_translation_corrected.json`; the file name is a scar).
>
> **2. A 1,491-feature "decode" was debunked.** An earlier version leaned on a high-dimensional feature
> decode that did not survive scrutiny. The translation result below is **independent of it** and does not
> use it. It has not been quietly reinstated.
>
> The lesson to carry forward: a result that is internally consistent is not thereby correct. Both errors
> produced clean, publishable-looking output.

In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
LOGS = ROOT / 'outputs' / 'logs'

# pagani_subtype_translation_corrected.json SUPERSEDES the earlier version. Do not read the old one.
pg = json.loads((LOGS / 'pagani_subtype_translation_corrected.json').read_text())
sp = json.loads((LOGS / 'pagani_spatial_subtype_routing.json').read_text())

print(f"NOTE FROM THE LOG: {pg['note']}")
print()
print(f"{pg['n_models']} autism mouse models: {pg['n_hyper']} hyper-connected, {pg['n_hypo']} hypo-connected")
print(f"leave-one-out subtype-assignment consistency: {pg['loo_consistency']}")

## 1. Route each subtype into human network space

For each subtype, take its mouse connectivity phenotype, push it through π, and read out the predicted
effect on each human functional network. Then compare with the observed human autism pattern.

The comparison to make is the **cross-correlation matrix**: does *predicted hyper* look like *observed
hyper*, and does it look *unlike* observed hypo? A prediction that correlates with both is not a
prediction.

In [ ]:
tr = pg['translation']
nets = tr['human_networks']
cc = tr['cross_correlation']

print(f'human networks: {nets}')
print()
print('cross-correlation of PREDICTED (mouse → human through π) with OBSERVED human autism patterns:')
print()
print(f"{'':<22} {'obs hyper':>12} {'obs hypo':>12}")
print('-' * 48)
for pred in ('hyper', 'hypo'):
    row = [cc.get(f'pred_{pred}__obs_{obs}', np.nan) for obs in ('hyper', 'hypo')]
    print(f"{'pred ' + pred:<22} {row[0]:>+12.2f} {row[1]:>+12.2f}")
print()
print(f"subtype-specific for HYPER: {tr['subtype_specific_hyper']}")
print(f"subtype-specific for HYPO : {tr['subtype_specific_hypo']}")

### Reading the matrix

The **hyper**-connected subtype translates *specifically*. Its prediction correlates with the observed
hyper pattern and anti-correlates with the observed hypo pattern. That is a real, directional result.

The **hypo**-connected subtype does **not**, and it fails in the worst possible way. Its prediction
correlates *positively* with the observed **hyper** pattern (+0.21) and *negatively* with the observed
**hypo** pattern (−0.13). The sign is wrong on both counts: predicted-hypo looks more like observed hyper
than like observed hypo. The log records this as `subtype_specific_hypo: false`.

**One of the two subtypes translates specifically and one does not.** Reporting only the one that worked
would be the easiest way to turn this into a false result, so we report both. The Discussion says "the
hyper-connected subtype", not "the subtypes".

In [ ]:
# ---------------- the cross-correlation matrix ----------------
M = np.array([[cc.get(f'pred_{p}__obs_{o}', np.nan) for o in ('hyper', 'hypo')]
              for p in ('hyper', 'hypo')])

fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(M, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks([0, 1]); ax.set_xticklabels(['observed\nhyper', 'observed\nhypo'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['predicted\nhyper', 'predicted\nhypo'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{M[i, j]:+.2f}', ha='center', va='center', fontsize=13, fontweight='bold',
                color='white' if abs(M[i, j]) > 0.55 else 'black')
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.set_label('correlation across human networks')
ax.set_title('Only the hyper-connected subtype\ntranslates subtype-specifically',
             fontweight='bold', loc='left', fontsize=10.5)
plt.show()

print('The top row discriminates: positive on its own subtype, negative on the other.')
print('The bottom row is BACKWARDS: predicted-hypo correlates positively with observed HYPER and')
print('negatively with observed HYPO. It points the wrong way rather than being uninformative.')
print()
print('We report this. Half of the analysis failed, and the failure is visible in the panel rather')
print('than cropped out of it. Note also that the whole matrix sits in the 0.2-0.4 range: even the')
print('working half is a modest correspondence rather than a strong one.')

## 2. The spatial routing (where in the human brain each subtype lands)

In [ ]:
print(f"matched parcels: {sp['n_matched_parcels']}")
print(f"contrast (hyper − hypo), predicted vs observed:")
print(f"  Pearson r  = {sp['contrast_pearson_r']:+.2f}")
print(f"  Spearman ρ = {sp['contrast_spearman_r']:+.2f}")
print(f"  empirical p = {sp['contrast_empirical_p']:.3f}  (null mean {sp['null_mean']:+.2f})")
print()
print('The predicted hyper−hypo contrast is positively related to the observed one, and clears the')
print('permutation null. But note the effect size: r = 0.17. That is a WEAK correspondence, and the')
print('permutation null it beats is a lenient one (null mean −0.63; the null is anti-correlated by')
print('construction, so beating it is not a high bar).')
print()
print('We do not oversell this. The network-level cross-correlation above is the result. This spatial')
print('contrast is corroborating detail rather than a second independent confirmation.')

In [ ]:
# ---------------- predicted vs observed, per human network ----------------
pred_h = np.array(sp['predicted']['hyper'], float)
obs_h = np.array(sp['observed']['hyper'], float)
labels = sp['human_networks']

x = np.arange(len(labels))
w = 0.38
fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.bar(x - w / 2, pred_h, w, color='#1b4f8a', label='predicted (mouse → human through π)', zorder=3)
ax.bar(x + w / 2, obs_h, w, color='#e08a2b', label='observed (human autism literature)', zorder=3)
ax.axhline(0, color='0.3', lw=1)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8.5)
ax.set_ylabel('connectivity effect (z)')
ax.legend(frameon=False, fontsize=9)
ax.set_title('Hyper-connected subtype routed into human network space\n'
             f"predicted vs observed hyper: r = {cc['pred_hyper__obs_hyper']:+.2f}; "
             f"vs observed hypo: r = {cc['pred_hyper__obs_hypo']:+.2f}",
             fontweight='bold', loc='left', fontsize=10.5)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
plt.show()

## 3. What this is, and what it is not

**What it is.** A demonstration that a calibrated, confidence-graded coupling lets you take a *mouse*
phenotype (one with no human counterpart by construction, because these are genetic mouse models) and
state which *human* functional networks it implicates, with a direction that can be checked against the
human literature. That is the use-case the method exists to serve.

**What it is not.**

- It is not a Results claim. It sits in the Discussion as an application.
- It is not a claim about **both** subtypes. Only the hyper-connected one translates specifically. The
  hypo-connected prediction points the wrong way, and we say so rather than dropping it.
- Even the working half is modest: +0.35 on its own subtype, −0.25 on the other. This is a directional
  result rather than a strong one.
- It does not use the 1,491-feature decode, which was debunked and is not reinstated.
- The spatial contrast (r = 0.17) is weak and beats only a lenient null. It corroborates. It does not
  independently confirm.

**Reproduces**

| output | script |
|---|---|
| `pagani_subtype_translation_corrected.json` | `experiments/pagani_2026_autism/` (supersedes the inverted-label version) |
| `pagani_spatial_subtype_routing.json` | same |
| per-model NIfTIs | **still outstanding** (see the project notes) |